In [5]:
import urllib.parse, urllib.request
import xml.etree.ElementTree as ET
import csv
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.fs as fs
import pyarrow.compute as pc
import time
import os
import datetime as dt


today = dt.date.today()
last_day = today.replace(day=1) - dt.timedelta(days=1)
first_day = last_day.replace(day=1)
BACKFILL_RANGE = (first_day.isoformat(), last_day.isoformat())

BASE_URL = "http://export.arxiv.org/api/query?"
HEADERS = {"User-Agent": "MyArxivClient/1.0 (martyshort52@gmail.com)"}
PAGE_SIZE = 100

# add opensearch to max out rows to write
ns = {
    'atom': 'http://www.w3.org/2005/Atom',
    'opensearch': 'http://a9.com/-/spec/opensearch/1.1/',
}

# lay out schema for pq
schema = pa.schema([
    ('id', pa.string()),
    ('title', pa.string()),
    ('author', pa.string()),
    ('category', pa.string()),
    ('abstract', pa.string()),
    ('published', pa.string()),
    ('url', pa.string()),
])

# loop over each search page
def fetch_page(q, start, retries=10):
    params = {'search_query': q, 'start': start, 'max_results': PAGE_SIZE}
    url = BASE_URL + urllib.parse.urlencode(params)
    for attempt in range(retries):
        try:
            req = urllib.request.Request(url, headers=HEADERS)
            with urllib.request.urlopen(req) as resp:
                return ET.fromstring(resp.read().decode('utf-8'))
        except urllib.error.HTTPError as e:
            if e.code in (429, 500, 503) and attempt < retries - 1:
                # arXiv sends 429 when we're going too fast; back off harder than 500/503
                wait = int(e.headers.get('Retry-After', min(60, 15 * (attempt + 1))))
                time.sleep(wait)
                continue
            raise

# date is a plain date (no time component), so %H%M always renders as 0000 -
# append 2359 to q_end explicitly or the range misses the whole last day
q_start = first_day.strftime('%Y%m%d0000')
q_end = last_day.strftime('%Y%m%d2359')
q = f'all:"information retrieval" AND submittedDate:[{q_start} TO {q_end}]'

root = fetch_page(q, 0)
total_results = int(root.find('opensearch:totalResults', ns).text)
entries = root.findall('atom:entry', ns)

start = PAGE_SIZE
while start < total_results:
    time.sleep(3)
    page_root = fetch_page(q, start)
    entries.extend(page_root.findall('atom:entry', ns))
    start += PAGE_SIZE

print(f"{first_day} to {last_day}: {len(entries)}/{total_results} entries")

#create empty list
ids, titles, authors, categories, abstracts, published, urls = [], [], [], [], [], [], []

# loop over each entry/paper
for entry in entries:

    # fill in the list of columns based on tag
    ids.append(entry.find('atom:id', ns).text.strip())
    titles.append(entry.find('atom:title', ns).text.strip().replace('\n', ' '))
    abstracts.append(entry.find('atom:summary', ns).text.strip().replace('\n', ' '))
    published.append(entry.find('atom:published', ns).text.strip())

    # add handling for multiple author tags
    author_names = [a.find('atom:name', ns).text for a in entry.findall('atom:author', ns)]
    authors.append('; '.join(author_names))

    # add handling for multiple category tags
    category_terms = [c.attrib['term'] for c in entry.findall('atom:category', ns)]
    categories.append('; '.join(category_terms))

    pdf_link_el = entry.find("atom:link[@title='pdf']", ns)
    urls.append(pdf_link_el.attrib['href'] if pdf_link_el is not None else None)

#create pq table using them lists
table = pa.Table.from_arrays(
    [
        pa.array(ids, type=pa.string()),
        pa.array(titles, type=pa.string()),
        pa.array(authors, type=pa.string()),
        pa.array(categories, type=pa.string()),
        pa.array(abstracts, type=pa.string()),
        pa.array(published, type=pa.string()),
        pa.array(urls, type=pa.string()),
    ],
    schema=schema,
)

table = table.set_column(
    table.schema.get_field_index('published'),
    'published',
    # parquet has no seconds-resolution timestamp, so backfill_arxiv.py's output
    # round-trips through S3 as timestamp[ms] - match that dtype upstream here
    # instead of casting existing_table after reading it back.
    pc.strptime(table.column('published'), format='%Y-%m-%dT%H:%M:%SZ', unit='ms'),
)
table = table.sort_by([('published', 'ascending')])

# push to s3
s3_filesystem = fs.S3FileSystem(region= "ap-southeast-1")

s3_path = "arvix-db/raw-arxiv/raw-arvix-entries.parquet"

# append to existing backfill instead of overwriting it
existing_file_info = s3_filesystem.get_file_info(s3_path)
if existing_file_info.type != fs.FileType.NotFound:
    existing_table = pq.read_table(s3_path, filesystem=s3_filesystem)
    table = pa.concat_tables([existing_table, table])
    table = table.sort_by([('published', 'ascending')])

# sandbox: writing to s3 is disabled until this is verified - reads are fine
# pq.write_table(table, s3_path, filesystem=s3_filesystem)
#
# file_info = s3_filesystem.get_file_info(s3_path)
# if file_info.type != fs.FileType.NotFound:
#     print(f"Backfill completed. raw-arvix-entries.parquet has been uploaded to s3://{s3_path}")
# else:
#     print(f"Backfill failed. raw-arvix-entries.parquet not found at s3://{s3_path}")

2026-08-01 to 2026-08-31: 540/540 entries


In [3]:
# pull all entries published on 2026-08-31 straight from the arXiv API (no S3)
day_start = dt.date(2026, 8, 31)
day_end = dt.date(2026, 8, 31)
q_start = day_start.strftime('%Y%m%d0000')
q_end = day_end.strftime('%Y%m%d2359')
q = f'all:"information retrieval" AND submittedDate:[{q_start} TO {q_end}]'

root = fetch_page(q, 0)
total_results = int(root.find('opensearch:totalResults', ns).text)
day_entries = root.findall('atom:entry', ns)

start = PAGE_SIZE
while start < total_results:
    time.sleep(3)
    page_root = fetch_page(q, start)
    day_entries.extend(page_root.findall('atom:entry', ns))
    start += PAGE_SIZE

print(f"2026-08-31: {len(day_entries)}/{total_results} entries")

day_rows = []
for entry in day_entries:
    author_names = [a.find('atom:name', ns).text for a in entry.findall('atom:author', ns)]
    category_terms = [c.attrib['term'] for c in entry.findall('atom:category', ns)]
    pdf_link_el = entry.find("atom:link[@title='pdf']", ns)
    day_rows.append({
        'id': entry.find('atom:id', ns).text.strip(),
        'title': entry.find('atom:title', ns).text.strip().replace('\n', ' '),
        'author': '; '.join(author_names),
        'category': '; '.join(category_terms),
        'abstract': entry.find('atom:summary', ns).text.strip().replace('\n', ' '),
        'published': entry.find('atom:published', ns).text.strip(),
        'url': pdf_link_el.attrib['href'] if pdf_link_el is not None else None,
    })

aug31_df = pd.DataFrame(day_rows)
aug31_df

2026-08-31: 37/37 entries


,id,title,author,category,abstract,published,url
0,http://arxiv.org/abs/2608.30949v1,MULTI3IR: A Benchmark for Multi-perspective Mu...,Seokwon Song; Sohyeon Kim; Gunhee Kim,cs.IR,Information retrieval (IR) increasingly target...,2026-08-31T15:19:26Z,https://arxiv.org/pdf/2608.30949v1
1,http://arxiv.org/abs/2608.31150v1,Local Private Information Retrieval for Graph-...,Shreya Meel; Mohamed Nomeir; Sennur Ulukus,cs.IT; cs.CR; cs.DB; cs.NI; eess.SP,We rethink the definition of privacy in multi-...,2026-08-31T17:50:47Z,https://arxiv.org/pdf/2608.31150v1
2,http://arxiv.org/abs/2608.31115v1,InsightToast: Proactive Information Retrieval ...,Mohammad Abolnejadian; Matthew Brehmer,cs.HC; cs.IR,Missing institutional context during meetings ...,2026-08-31T17:22:43Z,https://arxiv.org/pdf/2608.31115v1
3,http://arxiv.org/abs/2608.30823v1,Vocal Music under Phoneme-Conditional Analysis,Hayoon Kim; Kyogu Lee,cs.SD; cs.CL,The vocal music of each language carries a dis...,2026-08-31T14:03:36Z,https://arxiv.org/pdf/2608.30823v1
4,http://arxiv.org/abs/2608.30426v1,Learning to Reason and Use Tools through Unsup...,Markel Ferro; Oier Lopez de Lacalle,cs.CL,Current dialogue systems struggle with dynamic...,2026-08-31T08:22:26Z,https://arxiv.org/pdf/2608.30426v1
5,http://arxiv.org/abs/2609.00479v1,EGT-KG: Evidence-Grounded Typed KG Retrieval f...,Muran Yu; Jiechao Gao; Yuandong Pan; Barney H....,cs.AI,"For emerging scientific research domains, loca...",2026-08-31T23:27:57Z,https://arxiv.org/pdf/2609.00479v1
6,http://arxiv.org/abs/2608.30917v1,Intrinsic Scatterer Representation for Forward...,Ziyu Yue; Feng Xu,eess.SP,Forward modeling of scattering centers of rada...,2026-08-31T14:58:26Z,https://arxiv.org/pdf/2608.30917v1
7,http://arxiv.org/abs/2608.30291v1,PEARL: Front-Loading Relational Chains for Mul...,Subeen Ho; Hyeongu Kang; SeongKu Kang; Susik Yoon,cs.IR,While large language models (LLMs) have shown ...,2026-08-31T05:56:40Z,https://arxiv.org/pdf/2608.30291v1
8,http://arxiv.org/abs/2608.30606v1,Generative Retrieval for E-commerce: Jointly L...,Songtao Fang; Zihao Xu; Shaowei Wei; Jin Zhang...,cs.IR; cs.AI,With the development of large language models ...,2026-08-31T11:18:52Z,https://arxiv.org/pdf/2608.30606v1
9,http://arxiv.org/abs/2609.01654v1,MELON: A Large-Scale Dataset for Multi-Event T...,Chan Hur; SeungWoo Song; Jeong-hun Hong; Won J...,cs.IR,Existing text-video retrieval datasets primari...,2026-08-31T08:20:35Z,https://arxiv.org/pdf/2609.01654v1


In [4]:
len(table)

12070

In [6]:
len(table)

12107